# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print Dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, fields, and columns by their @id
record_sets = metadata.recordSet

print('Record Sets:')
recordset_ids = []
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    recordset_ids.append(rs['@id'])
    if 'field' in rs:
        print('    Fields:')
        for field in rs['field']:
            print(f"      Field @id: {field['@id']} (name: {field.get('name', '')})")
            if 'column' in field:
                print('        Columns:')
                for col in field['column']:
                    print(f"        Column @id: {col['@id']} (name: {col.get('name', '')})")

# Optionally, print example records from a record set
if recordset_ids:
    print('\nExample records from the first record set:')
    for x in dataset.records(record_set=recordset_ids[0]):
        print(x)
        break  # Print just the first example record

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the `@id` fields discovered above.

In [ ]:
# Extract all record sets into pandas DataFrames
dataframes = {}

for record_set_id in recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFor RecordSet @id: {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))

# Choose the first record set for further analysis
example_record_set = recordset_ids[0]
df_example = dataframes[example_record_set]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Identify numeric fields from the first record set
numeric_fields = []
for field in record_sets[0]['field']:
    if 'dataType' in field and field['dataType'] in ('schema:Float', 'schema:Integer'):
        numeric_fields.append(field['@id'])

print(f"Numeric field candidates (by @id): {numeric_fields}")

# Select a numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    numeric_field_name = record_sets[0]['field'][0].get('name', numeric_field_id)
else:
    numeric_field_id = df_example.select_dtypes('number').columns[0] if len(df_example.select_dtypes('number').columns) > 0 else df_example.columns[0]
    numeric_field_name = numeric_field_id

threshold = 10

# Filtering records where numeric field > threshold
filtered_df = df_example[df_example[numeric_field_name] > threshold]
print(f"Filtered records with {numeric_field_name} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_name}_normalized"] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()
print(f"Normalized {numeric_field_name} for filtered records:")
print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

# Try grouping by a categorical column
group_field_candidates = [col for col in df_example.columns if df_example[col].dtype == 'object' and col != numeric_field_name]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped = filtered_df.groupby(group_field)[numeric_field_name].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field_name}):")
    print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df_example[numeric_field_name].dropna(), bins=20, kde=True)
plt.xlabel(numeric_field_name)
plt.title(f"Distribution of {numeric_field_name}")
plt.show()

if group_field_candidates:
    group_field = group_field_candidates[0]
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df_example[group_field], y=df_example[numeric_field_name])
    plt.title(f"{numeric_field_name} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, review, and analyze the FAIR² Mental Health Survey dataset using `mlcroissant`, referencing all entities by their `@id`.

- We explored available record sets, fields, and columns.
- Data was processed and visualized, with numeric fields normalized and grouped by key attributes.
- All operations referenced entities dynamically using their `@id` for reproducibility.
You can extend this workflow to more advanced analytical tasks or integrate additional machine learning steps as needed for your analysis.